In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import random

from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from bait.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils
from bait.core import bait_utils
from bait.core.bait_prompts import FILE_FORMATS, CONTEXT_SIZE, get_generate_prompt

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/bait'
data_dir = f'{work_dir}/data'
in_dir = f'{data_dir}/create_contexts'
out_dir = f'{data_dir}/main_experiments'

dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

In [ ]:
def mix_contexts(contexts_fact_dict: dict, contexts_counter_dict: dict, ext_n_fact: int, ext_n_counter: int):
    if ext_n_fact <= len(contexts_fact_dict) and ext_n_counter <= len(contexts_counter_dict):
        ext_contexts_fact = random.sample(list(contexts_fact_dict.values()), ext_n_fact)
        ext_contexts_counter = random.sample(list(contexts_counter_dict.values()), ext_n_counter)
        
        mixed_contexts = ext_contexts_fact + ext_contexts_counter
        random.shuffle(mixed_contexts)

        return mixed_contexts
    
    return None

In [ ]:
def add_prompts(question: str, answer: str, contexts_fact: dict, contexts_counter: dict, prompts: list, answers: list, states: list):
    for file_format in FILE_FORMATS:
        contexts_fact_dict = contexts_fact[file_format]
        contexts_counter_dict = contexts_counter[file_format]

        for i in range(CONTEXT_SIZE):
            mixed_contexts = mix_contexts(contexts_fact_dict, contexts_counter_dict, i, CONTEXT_SIZE-1-i)

            if mixed_contexts is None:
                prompts.append(get_generate_prompt(question))
                answers.append(answer)
                states.append(False)
            else:
                prompts.append(get_generate_prompt(question, mixed_contexts))
                answers.append(answer)
                states.append(True)

In [ ]:
def experiment_context_ratio(model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizerFast, zero_shot, datas, batch_size=1):
    data_size = len(datas)
    cnts = {}

    for i, datas_batch in enumerate(container_utils.chunks(datas, batch_size)):
        prompts_batch, answers_batch, states_batch = [], [], []

        for data in datas_batch:
            question = data['question']
            answer_fact = data['answer_fact']
            answer_counter = data['answer_counter']
            contexts_fact = data['contexts_fact']
            contexts_counter = data['contexts_counter']
            
            add_prompts(question, answer_fact, contexts_fact, contexts_counter, prompts_batch, answers_batch, states_batch)
        
        # print(f'prompts_batch size : {len(prompts_batch)}')
        # print(f'answers_batch size : {len(answers_batch)}\n')

        generated_texts = model_utils.get_generated_texts(
            model, tokenizer, device,
            prompts_batch, max_seq_length, max_new_tokens
        )

        idx = -1
        for j in range(batch_size):
            for file_format in FILE_FORMATS:
                for k in range(CONTEXT_SIZE):
                    idx += 1

                    ext_n_fact = idx % CONTEXT_SIZE
                    ext_n_counter = CONTEXT_SIZE - 1 - ext_n_fact

                    # key 저장 용도
                    container_utils.add_str_int(cnts, f'{file_format}\t{ext_n_fact}\t{ext_n_counter}', 0)
                    container_utils.add_str_int(cnts, f'ALL\t{ext_n_fact}\t{ext_n_counter}', 0)

                    if not states_batch[idx]:
                        container_utils.add_str_int(cnts, f'{file_format}_skip\t{ext_n_fact}\t{ext_n_counter}', 1)
                        container_utils.add_str_int(cnts, f'ALL_skip\t{ext_n_fact}\t{ext_n_counter}', 1)
                    else:
                        if model_utils.is_correct(generated_texts[idx], answers_batch[idx])[1]:
                            container_utils.add_str_int(cnts, f'{file_format}\t{ext_n_fact}\t{ext_n_counter}', 1)
                            container_utils.add_str_int(cnts, f'ALL\t{ext_n_fact}\t{ext_n_counter}', 1)

        if (i+1) % 100 == 0:
            print(f'experiment_context_ratio() {(i+1)*batch_size} complet.')
    print(f'experiment_context_ratio() {data_size} complet.\n')





    file_formats = FILE_FORMATS + ['ALL']
    for file_format in file_formats:
        for i in range(CONTEXT_SIZE):
            cnt_key = f'{file_format}\t{CONTEXT_SIZE-1-i}\t{i}'
            cnt_skip_key = f'{file_format}_skip\t{CONTEXT_SIZE-1-i}\t{i}'

            if cnt_key in cnts.keys():
                cnt_value = cnts[cnt_key]

                if cnt_key.startswith('ALL'):
                    size = data_size * len(FILE_FORMATS)
                else:
                    size = data_size

                if cnt_skip_key in cnts.keys():
                    size -= cnts[cnt_skip_key]

                print(f'{cnt_key}\t{cnt_value}\t{cnt_value}/{size}\t{cnt_value/size}')
        print()

In [ ]:
model_names = ['Llama-3.2-3B', 'Llama-3.1-8B', 'Qwen2.5-3B', 'Qwen2.5-7B']

for model_name in model_names:
    model_name_or_path = bait_utils.get_model_name_or_path(model_name)
    
    model = model_utils.get_model(model_name_or_path, dtype, device=device, is_eval=True)

    # 평가/추론 시에는 반드시 'left' 패딩
    tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

    # model = None
    # tokenizer = None

    for zero_shot in ['fact', 'counter', 'other']:
        in_file_path = f'{in_dir}/{model_name}/bait_{model_name}_zero_shot_{zero_shot}_created_contexts.json'
        datas = json_utils.load_json(in_file_path)

        experiment_context_ratio(model, tokenizer, zero_shot, datas)

    del model
    del tokenizer
    common_utils.clear_gpu_memory()